# Federated Vitals Anomaly Detection — Temporal 1D-CNN

Trains a temporal 1D-CNN on patient vitals (HR, SpO2, awRR) with federated learning.

**Improvements over MLP baseline:**
- Temporal model (1D-CNN) processes full sequences instead of point-wise
- NaN handling via mask channels (not row dropping)
- Subject-aware non-IID Dirichlet split
- Proper train/test separation

**Methods:** Centralized, FedAvg, FedProx, FedBN

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup & GPU Check

In [ ]:
!pip install -q scikit-learn pyarrow

import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU"

In [ ]:
import os, glob, json, time, warnings
from collections import OrderedDict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')

# ── Reproducibility ──
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

## 2. Upload Data

Upload the `integrated_v2/` parquet files.  
**Option A:** Google Drive (recommended for large files)  
**Option B:** Direct upload

In [ ]:
# ── Option A: Google Drive (recommended) ──
# 1. Upload ntu_action_dataset/integrated_v2/*.parquet to your Drive
# 2. Run this cell

from google.colab import drive
drive.mount('/content/drive')

# Set this to your Drive path containing the parquet files:
DATA_DIR = '/content/drive/MyDrive/ntu_action_dataset/integrated_v2'

files = sorted(glob.glob(os.path.join(DATA_DIR, '*.parquet')))
print(f'Found {len(files)} parquet files')
assert len(files) > 0, f'No parquet files found in {DATA_DIR}! Check the path.'
for f in files:
    print(f'  {os.path.basename(f)}')

In [ ]:
# # ── Option B: Direct upload (uncomment if not using Drive) ──
# from google.colab import files as colab_files
# os.makedirs('data/integrated_v2', exist_ok=True)
# print('Upload all integrated_v2/*.parquet files:')
# uploaded = colab_files.upload()
# for name, content in uploaded.items():
#     with open(f'data/integrated_v2/{name}', 'wb') as f:
#         f.write(content)
# DATA_DIR = 'data/integrated_v2'
# print(f'Uploaded {len(uploaded)} files')

## 3. Configuration

In [ ]:
# ── All hyperparameters in one place ──
CONFIG = {
    # Data
    'vitals_channels': ['HR', 'SpO2', 'awRR'],
    'seq_len': 200,          # 100 seconds @ 0.5s dt
    'test_frac': 0.2,
    # Model
    'n_channels': 6,         # 3 vitals + 3 masks
    'n_classes': 6,
    # Training
    'centralized_epochs': 50,
    'batch_size': 32,
    'lr': 0.001,
    'weight_decay': 1e-4,
    # FL
    'num_clients': 5,
    'dirichlet_alpha': 0.5,
    'local_epochs': 2,
    'num_rounds': 30,
    'fedprox_mu': 0.01,
    # Seed
    'seed': SEED,
}

CLASS_NAMES = ['Bradycardia', 'High pressure', 'Low pressure',
               'Nitroglycerine', 'Normal', 'Tachycardia']

print('Config:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 4. Data Preprocessing

In [ ]:
%%time

# Load all parquet chunks
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.parquet')))
iv2 = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f'{iv2.shape[0]:,} rows, {iv2.sequence_id.nunique()} sequences, '
      f'{iv2.subject_id.nunique()} subjects')

# Sort by sequence + time
iv2 = iv2.sort_values(['sequence_id', 'timestamp_s']).reset_index(drop=True)

# ── Build temporal tensors ──
VITALS = CONFIG['vitals_channels']
MASKS  = [f'{v}_mask' for v in VITALS]
SEQ_LEN = CONFIG['seq_len']
N_CH = CONFIG['n_channels']
class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}

sequences, labels, subject_ids = [], [], []

for seq_id, grp in iv2.groupby('sequence_id'):
    vitals = grp[VITALS].values.astype(np.float32)
    masks  = grp[MASKS].values.astype(np.float32)
    vitals = np.nan_to_num(vitals, nan=0.0)
    combined = np.concatenate([vitals, masks], axis=1)  # (T, 6)

    T = combined.shape[0]
    if T >= SEQ_LEN:
        start = (T - SEQ_LEN) // 2  # center crop
        combined = combined[start:start + SEQ_LEN]
    else:
        pad = np.zeros((SEQ_LEN - T, N_CH), dtype=np.float32)
        combined = np.concatenate([combined, pad], axis=0)

    sequences.append(combined)
    labels.append(class_to_idx[grp['class_name'].iloc[0]])
    subject_ids.append(grp['subject_id'].iloc[0])

X = np.stack(sequences).transpose(0, 2, 1)  # (N, 6, SEQ_LEN)
y = np.array(labels, dtype=np.int64)
subjects = np.array(subject_ids)

# ── Per-channel normalization (valid values only) ──
n_vitals = len(VITALS)
for ch in range(n_vitals):
    vals = X[:, ch, :]
    mask_vals = X[:, ch + n_vitals, :]
    valid = vals[mask_vals > 0.5]
    if len(valid) > 0:
        mu, sigma = valid.mean(), valid.std() + 1e-8
        X[:, ch, :] = np.where(mask_vals > 0.5, (vals - mu) / sigma, 0.0)

print(f'Tensor: X={X.shape}, y={y.shape}')
print(f'Classes: {dict(zip(CLASS_NAMES, np.bincount(y)))}')
print(f'Subjects: {len(np.unique(subjects))}')

# Free memory
del iv2, sequences

In [ ]:
# ── Subject-aware train/test split ──
rng = np.random.RandomState(SEED)
unique_subj = np.unique(subjects)
rng.shuffle(unique_subj)
n_test = int(len(unique_subj) * CONFIG['test_frac'])
test_subj = set(unique_subj[:n_test])

train_mask = np.array([s not in test_subj for s in subjects])
test_mask = ~train_mask

X_train, y_train, subj_train = X[train_mask], y[train_mask], subjects[train_mask]
X_test, y_test, subj_test    = X[test_mask],  y[test_mask],  subjects[test_mask]

print(f'Train: {X_train.shape[0]} seqs ({len(unique_subj) - n_test} subjects)')
print(f'Test:  {X_test.shape[0]} seqs ({n_test} subjects)')
print(f'Train class dist: {np.bincount(y_train).tolist()}')
print(f'Test  class dist: {np.bincount(y_test).tolist()}')

## 5. Model Definition

In [ ]:
class VitalsTemporalCNN(nn.Module):
    """
    1D-CNN for vitals time series classification.
    Input:  (batch, 6, 200) — 3 vitals + 3 masks, 200 timesteps
    Output: (batch, 6) — 6 class logits
    """
    def __init__(self, in_channels=6, n_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


model = VitalsTemporalCNN().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'VitalsTemporalCNN: {n_params:,} parameters')
print(model)

## 6. Training Utilities

In [ ]:
def make_loader(X, y, batch_size=CONFIG['batch_size'], shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    model.eval()
    correct = total = 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return correct / total, np.array(all_preds), np.array(all_labels)


def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE,
                    proximal_mu=0.0, global_params=None):
    model.train()
    epoch_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        if proximal_mu > 0.0 and global_params is not None:
            prox = sum(((p - gp) ** 2).sum()
                       for p, gp in zip(model.parameters(), global_params))
            loss += (proximal_mu / 2.0) * prox
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    return epoch_loss / len(loader.dataset)

## 7. Centralized Training (Upper Bound)

In [ ]:
%%time

model_cent = VitalsTemporalCNN().to(DEVICE)
train_loader = make_loader(X_train, y_train)
test_loader  = make_loader(X_test, y_test, shuffle=False)

optimizer = optim.Adam(model_cent.parameters(), lr=CONFIG['lr'],
                       weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                  T_max=CONFIG['centralized_epochs'])
criterion = nn.CrossEntropyLoss()

hist_cent = {'train_acc': [], 'test_acc': [], 'loss': []}
best_cent_acc = 0
best_cent_state = None

for epoch in range(1, CONFIG['centralized_epochs'] + 1):
    loss = train_one_epoch(model_cent, train_loader, optimizer, criterion)
    scheduler.step()
    train_acc, _, _ = evaluate(model_cent, train_loader)
    test_acc, _, _  = evaluate(model_cent, test_loader)
    hist_cent['loss'].append(loss)
    hist_cent['train_acc'].append(train_acc)
    hist_cent['test_acc'].append(test_acc)
    if test_acc > best_cent_acc:
        best_cent_acc = test_acc
        best_cent_state = {k: v.cpu().clone() for k, v in model_cent.state_dict().items()}
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}: loss={loss:.4f}  train={train_acc:.1%}  test={test_acc:.1%}')

print(f'\nBest centralized test accuracy: {best_cent_acc:.1%}')

## 8. Federated Learning Utilities

In [ ]:
def dirichlet_split(y_train, subjects_train, n_clients, alpha, n_classes=6, seed=42):
    """
    Label-based Dirichlet non-IID split with subject-awareness.
    Subjects are kept intact — all sequences from one subject go to same client.
    """
    rng = np.random.RandomState(seed)

    # Group subjects by dominant class
    subject_class = {}
    for subj in np.unique(subjects_train):
        mask = subjects_train == subj
        subject_class[subj] = np.bincount(y_train[mask], minlength=n_classes).argmax()

    # Per-class Dirichlet partition of subjects
    subject_assignment = {}
    for cls in range(n_classes):
        cls_subjects = [s for s, c in subject_class.items() if c == cls]
        rng.shuffle(cls_subjects)
        if not cls_subjects:
            continue
        props = rng.dirichlet([alpha] * n_clients)
        props = np.maximum(props, 0.05)
        props /= props.sum()
        splits = np.array_split(cls_subjects,
                                (np.cumsum(props)[:-1] * len(cls_subjects)).astype(int))
        for cid, subj_list in enumerate(splits):
            for s in subj_list:
                subject_assignment[s] = cid

    client_indices = {i: [] for i in range(n_clients)}
    for idx, s in enumerate(subjects_train):
        client_indices[subject_assignment.get(s, 0)].append(idx)

    print(f'Client sizes: {[len(v) for v in client_indices.values()]}')
    for cid in range(n_clients):
        dist = np.bincount(y_train[client_indices[cid]], minlength=n_classes)
        print(f'  Client {cid}: {len(client_indices[cid])} seqs, dist={dist.tolist()}')
    return client_indices


def get_params(model):
    return [p.clone().detach() for p in model.parameters()]


def set_params(model, params):
    for p, new_p in zip(model.parameters(), params):
        p.data.copy_(new_p)


def fedavg_aggregate(params_list, sizes):
    total = sum(sizes)
    return [sum(params_list[c][i] * (sizes[c] / total)
                for c in range(len(params_list)))
            for i in range(len(params_list[0]))]


def fedbn_aggregate(state_dicts, sizes):
    total = sum(sizes)
    n = len(state_dicts)
    avg = OrderedDict()
    for key in state_dicts[0]:
        if 'bn' in key or 'num_batches_tracked' in key:
            avg[key] = state_dicts[0][key].clone()
        else:
            avg[key] = sum(state_dicts[c][key] * (sizes[c] / total) for c in range(n))
    return avg

print('FL utilities ready.')

In [ ]:
def train_federated(X_train, y_train, subjects_train, X_test, y_test,
                    mode='fedavg', num_rounds=30, mu=0.01):
    """Federated training loop. mode: 'fedavg' | 'fedprox' | 'fedbn'"""
    n_clients = CONFIG['num_clients']
    local_epochs = CONFIG['local_epochs']
    lr = CONFIG['lr']

    label = mode.upper()
    if mode == 'fedprox': label = f'FedProx (μ={mu})'
    print(f'\n{"="*60}')
    print(f' {label} — {n_clients} clients, α={CONFIG["dirichlet_alpha"]}, {num_rounds}R')
    print(f'{"="*60}')

    # Split data across clients
    client_indices = dirichlet_split(y_train, subjects_train, n_clients,
                                     CONFIG['dirichlet_alpha'])
    client_loaders = {}
    client_sizes = {}
    for cid in range(n_clients):
        idxs = client_indices[cid]
        client_loaders[cid] = make_loader(X_train[idxs], y_train[idxs])
        client_sizes[cid] = len(idxs)

    test_loader = make_loader(X_test, y_test, shuffle=False)
    global_model = VitalsTemporalCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    client_bn_states = {cid: None for cid in range(n_clients)}

    history = {'global_acc': [], 'client_accs': []}
    best_acc = 0
    best_state = None

    for rnd in range(1, num_rounds + 1):
        all_params, all_states, all_sizes = [], [], []
        client_accs = []

        for cid in range(n_clients):
            local = VitalsTemporalCNN().to(DEVICE)

            if mode == 'fedbn' and client_bn_states[cid] is not None:
                gs = global_model.state_dict()
                ls = client_bn_states[cid]
                merged = OrderedDict()
                for key in gs:
                    merged[key] = ls[key] if ('bn' in key or 'num_batches_tracked' in key) else gs[key]
                local.load_state_dict(merged)
            else:
                local.load_state_dict(global_model.state_dict())

            gp = [p.clone().detach().to(DEVICE) for p in global_model.parameters()] if mode == 'fedprox' else None

            opt = optim.Adam(local.parameters(), lr=lr, weight_decay=CONFIG['weight_decay'])
            for _ in range(local_epochs):
                train_one_epoch(local, client_loaders[cid], opt, criterion,
                                proximal_mu=mu if mode == 'fedprox' else 0.0,
                                global_params=gp)

            all_params.append(get_params(local))
            all_states.append(OrderedDict({k: v.cpu() for k, v in local.state_dict().items()}))
            all_sizes.append(client_sizes[cid])

            if mode == 'fedbn':
                client_bn_states[cid] = all_states[-1]

            c_acc, _, _ = evaluate(local, client_loaders[cid])
            client_accs.append(c_acc)

        # Aggregate
        if mode == 'fedbn':
            global_model.load_state_dict(fedbn_aggregate(all_states, all_sizes))
        else:
            set_params(global_model, fedavg_aggregate(all_params, all_sizes))

        g_acc, _, _ = evaluate(global_model, test_loader)
        history['global_acc'].append(g_acc)
        history['client_accs'].append(client_accs)

        if g_acc > best_acc:
            best_acc = g_acc
            best_state = {k: v.cpu().clone() for k, v in global_model.state_dict().items()}

        if rnd % 5 == 0 or rnd == 1:
            ca = ' '.join(f'{a:.0%}' for a in client_accs)
            print(f'  R{rnd:3d}: global={g_acc:.1%}  clients=[{ca}]')

    print(f'Best {mode} test accuracy: {best_acc:.1%}')

    # Final eval with best weights
    global_model.load_state_dict(best_state)
    global_model.to(DEVICE)
    _, preds, labels = evaluate(global_model, test_loader)

    return {'best_acc': best_acc, 'history': history,
            'preds': preds, 'labels': labels, 'state': best_state}

## 9. Run All FL Methods

In [ ]:
%%time

results = {}

# Store centralized results
model_cent.load_state_dict(best_cent_state)
model_cent.to(DEVICE)
_, cent_preds, cent_labels = evaluate(model_cent, test_loader)
results['Centralized'] = {
    'best_acc': best_cent_acc,
    'history': hist_cent,
    'preds': cent_preds,
    'labels': cent_labels,
}

# FedAvg
results['FedAvg'] = train_federated(
    X_train, y_train, subj_train, X_test, y_test,
    mode='fedavg', num_rounds=CONFIG['num_rounds'])

# FedProx
results['FedProx'] = train_federated(
    X_train, y_train, subj_train, X_test, y_test,
    mode='fedprox', num_rounds=CONFIG['num_rounds'], mu=CONFIG['fedprox_mu'])

# FedBN
results['FedBN'] = train_federated(
    X_train, y_train, subj_train, X_test, y_test,
    mode='fedbn', num_rounds=CONFIG['num_rounds'])

## 10. Results Summary

In [ ]:
print('=' * 50)
print(' RESULTS SUMMARY')
print('=' * 50)
for name, res in results.items():
    print(f'  {name:15s}: {res["best_acc"]:.1%}')

# Comparison with old MLP baseline
print('\n--- vs Old MLP (federated_2.py) ---')
print('  Old Centralized (MLP, eval-on-train): ~79%')
print('  Old FedAvg-10 (MLP):                  ~59%')
print('  Old FedProx (MLP):                    ~35%')

## 11. Visualizations

In [ ]:
# ── Training curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Vitals Anomaly Detection — Temporal 1D-CNN', fontsize=14, fontweight='bold')

# Centralized curve
ax = axes[0]
ax.plot(hist_cent['test_acc'], label='Centralized (test)', linewidth=2)
ax.plot(hist_cent['train_acc'], '--', label='Centralized (train)', alpha=0.6)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Centralized Training')
ax.legend()
ax.grid(True, alpha=0.3)

# FL curves
ax = axes[1]
for name in ['FedAvg', 'FedProx', 'FedBN']:
    if name in results:
        ax.plot(results[name]['history']['global_acc'], label=name, linewidth=2)
ax.set_xlabel('Round')
ax.set_ylabel('Global Test Accuracy')
ax.set_title('Federated Training')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Accuracy comparison bar chart ──
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
accs = [results[n]['best_acc'] for n in names]
colors = ['#1565c0', '#43a047', '#e65100', '#6a1b9a']
bars = ax.bar(names, accs, color=colors[:len(names)])
ax.set_ylabel('Best Test Accuracy')
ax.set_title('Vitals Classification — Method Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.1%}',
            ha='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ──
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]

for i, (name, res) in enumerate(results.items()):
    ax = axes[i]
    cm = confusion_matrix(res['labels'], res['preds'])
    ax.imshow(cm, cmap='Blues', interpolation='nearest')
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels([c[:6] for c in CLASS_NAMES], rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_yticklabels([c[:6] for c in CLASS_NAMES], fontsize=7)
    ax.set_title(f'{name}\n{res["best_acc"]:.1%}', fontweight='bold')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            ax.text(c, r, str(cm[r,c]), ha='center', va='center', fontsize=7,
                    color='white' if cm[r,c] > cm.max()*0.5 else 'black')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-class F1 report for best method ──
best_name = max(results, key=lambda n: results[n]['best_acc'])
best_res = results[best_name]
print(f'Classification report for {best_name} ({best_res["best_acc"]:.1%}):\n')
print(classification_report(best_res['labels'], best_res['preds'],
                            target_names=CLASS_NAMES, digits=3))

## 12. Save Models & Download

In [ ]:
os.makedirs('outputs_vitals', exist_ok=True)

# Save all model checkpoints
for name, res in results.items():
    if 'state' in res:
        path = f'outputs_vitals/{name.lower()}_best.pt'
        torch.save(res['state'], path)
        print(f'Saved: {path}')

# Save centralized
torch.save(best_cent_state, 'outputs_vitals/centralized_best.pt')
print('Saved: outputs_vitals/centralized_best.pt')

# Save results JSON
summary = {
    name: {'best_acc': float(res['best_acc'])}
    for name, res in results.items()
}
summary['config'] = CONFIG
with open('outputs_vitals/results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('Saved: outputs_vitals/results.json')

print('\nAll saved. Download from outputs_vitals/ or copy to Drive.')

In [ ]:
# ── Copy to Drive (optional) ──
# DRIVE_OUT = '/content/drive/MyDrive/ntu_action_dataset/vitals_results'
# os.makedirs(DRIVE_OUT, exist_ok=True)
# !cp outputs_vitals/* "{DRIVE_OUT}/"
# !cp *.png "{DRIVE_OUT}/"
# print(f'Copied to {DRIVE_OUT}')

## 13. Summary & Next Steps

| Method | Best Test Acc | Notes |
|--------|:---:|---|
| **Old MLP (eval-on-train)** | ~79% | Broken baseline |
| **Old FedAvg MLP** | ~59% | Point-wise, IID split |
| **Centralized 1D-CNN** | TBD | Upper bound |
| **FedAvg 1D-CNN** | TBD | Non-IID Dirichlet α=0.5 |
| **FedProx 1D-CNN** | TBD | μ=0.01 |
| **FedBN 1D-CNN** | TBD | Local BatchNorm |

**Key improvements:**
1. Temporal model processes full sequences (not individual rows)
2. Mask channels handle missing data (no row dropping → 100% data used)
3. Subject-aware split prevents data leakage
4. Proper train/test separation

**Next:** Update combined emergency system demo with these results.